# Partial integration of a polynomial over part of the domain

A function defined on a domain $[x_{inf}, x_{sup}]$, described with the natural coordinate $\xi \in [-1, 1]$ of the whole domain,

$$x = x_{inf} + \frac{(\xi + 1)(x_{sup} - x_{inf})}{2},$$

often has to be integrated over a part $[x_1, x_2]$ of the domain only, e.g. a load acting on part of a panel. This notebook shows that the Gauss-Legendre quadrature can be mapped onto the interval $[\xi_1, \xi_2]$ of the natural coordinate that corresponds to $[x_1, x_2]$, with the same accuracy as a quadrature in $x$. It uses NumPy and SciPy only, not `panels`.

A Gauss-Legendre rule with $n$ points $\xi_i$ and weights $w_i$ integrates a polynomial of degree up to $2n - 1$ exactly over $[-1, 1]$. The points and weights come from `scipy.special.roots_legendre`.

In [ ]:
import numpy as np
from scipy.special import roots_legendre
from IPython.display import Markdown, display

n = 3
pts, ws = roots_legendre(n)
print('Gauss-Legendre points:', pts)
print('weights:              ', ws)

## The polynomial and its exact integral

The integrand is the polynomial of fourth degree

$$f(x) = 4 + 2x - 4x^2 + 2x^3 - 10x^4,$$

whose integral is known in closed form. With $n = 3$ points the rule is exact up to degree 5, so the quadrature must reproduce it to the round-off.

In [ ]:
def f(x):
    return 4 + 2*x - 4*x**2 + 2*x**3 - 10*x**4


def F(x):
    """Antiderivative of f"""
    return 4*x + x**2 - 4*x**3/3 + x**4/2 - 2*x**5


def exact(x1, x2):
    return F(x2) - F(x1)

## Quadrature in $x$

Over $[x_1, x_2]$ directly: $x = x_1 + (\xi + 1)(x_2 - x_1)/2$ and $\mathrm{d}x = \frac{x_2 - x_1}{2}\mathrm{d}\xi$,

$$\int_{x_1}^{x_2} f(x)\,\mathrm{d}x \approx \frac{x_2 - x_1}{2}\sum_i w_i f\left(x_1 + \frac{(\xi_i + 1)(x_2 - x_1)}{2}\right).$$

In [ ]:
def quad_x(x1, x2):
    x = x1 + (pts + 1)*(x2 - x1)/2
    return (x2 - x1)/2*np.sum(ws*f(x))

## Quadrature in the natural coordinate of the whole domain

When the integrand is evaluated as a function of $\xi$ of the whole domain, the part $[x_1, x_2]$ corresponds to

$$\xi_1 = 2\frac{x_1 - x_{inf}}{x_{sup} - x_{inf}} - 1, \qquad \xi_2 = 2\frac{x_2 - x_{inf}}{x_{sup} - x_{inf}} - 1,$$

and each Gauss point is mapped onto $[\xi_1, \xi_2]$ by $\xi' = \xi_1 + (\xi_i + 1)(\xi_2 - \xi_1)/2$. The integrand is evaluated at $\xi'$ and the weights are scaled by $(x_2 - x_1)/2$, the Jacobian of the mapping from $[-1, 1]$ to $[x_1, x_2]$.

In [ ]:
def f_xi(xi, xinf, xsup):
    """f as a function of the natural coordinate of the whole domain"""
    return f(xinf + (xi + 1)*(xsup - xinf)/2)


def quad_xi(x1, x2, xinf, xsup):
    xi1 = 2*(x1 - xinf)/(xsup - xinf) - 1
    xi2 = 2*(x2 - xinf)/(xsup - xinf) - 1
    xi = xi1 + (pts + 1)*(xi2 - xi1)/2
    return (x2 - x1)/2*np.sum(ws*f_xi(xi, xinf, xsup))

## Results

The whole domain is $[0, 10]$; the integrals over the whole domain and over the part $[4, 6]$ are compared with the exact values, with the relative difference $(\text{quadrature} - \text{exact})/\text{exact}$.

In [ ]:
xinf, xsup = 0., 10.
rows = []
for x1, x2 in ((0., 10.), (4., 6.)):
    ref = exact(x1, x2)
    a, b = quad_x(x1, x2), quad_xi(x1, x2, xinf, xsup)
    rows.append('| [%g, %g] | %.10g | %.10g | %.1e | %.10g | %.1e |'
                % (x1, x2, ref, a, (a - ref)/ref, b, (b - ref)/ref))
display(Markdown('\n'.join(['| interval | exact | quadrature in $x$ | rel. diff. | quadrature in $\\xi$ | rel. diff. |',
                            '|---|---|---|---|---|---|'] + rows)))

## Discussion

Both quadratures reproduce the exact integrals to the round-off, over the whole domain and over the part: mapping the Gauss points onto $[\xi_1, \xi_2]$ keeps the degree of exactness of the rule, since the mapping is linear. A part of the domain therefore needs no change of the functions written in the natural coordinate of the whole domain, only the mapped points and the scaled weights. The notebook [Partial integration of a matrix with Gauss-Legendre points](test_quadrature_partial_integration_matrix.ipynb) applies the same mapping to a matrix integrand split into several parts.